In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import roc_curve, auc, precision_recall_curve
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

FACES140K_DIR = '/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake'
WEIGHTS_GAN    = '/kaggle/input/datasets/dhathrikarthik/deepfake-classifier-weights/deepfake_classifier.pth'
WEIGHTS_GAN_FT = '/kaggle/input/datasets/dhathrikarthik/deepfake-classifier-finetuned/deepfake_classifier_finetuned.pth'
WEIGHTS_CELEB  = '/kaggle/input/datasets/dhathrikarthik/deepfake-celebdf-weights/deepfake_celebdf.pth'

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

class FaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.data = []
        for label_name, label in [('real', 0), ('fake', 1)]:
            folder = os.path.join(root_dir, label_name)
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.data.append((os.path.join(folder, fname), label))
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

test_dataset = FaceDataset(f'{FACES140K_DIR}/test', transform=transform)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"Test set size: {len(test_dataset)}")

def _load(weights_path, strip_prefix=False):
    model = models.efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    state = torch.load(weights_path, map_location=DEVICE)
    if strip_prefix:
        state = {k.replace('model.', '', 1): v for k, v in state.items()}
    model.load_state_dict(state)
    model.to(DEVICE).eval()
    return model

gan_model    = _load(WEIGHTS_GAN,    strip_prefix=True)
gan_model_ft = _load(WEIGHTS_GAN_FT, strip_prefix=False)
celeb_model  = _load(WEIGHTS_CELEB,  strip_prefix=False)
print("✅ All three models loaded")

# ── Collect per-image scores from all models + true labels ─────────────────
all_labels = []
all_fake_max = []        # ensemble max score per image
all_winner = []           # which model contributed the max: 0=gan, 1=gan_ft, 2=celeb

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        probs_gan    = torch.softmax(gan_model(images),    dim=1)[:, 1]
        probs_gan_ft = torch.softmax(gan_model_ft(images), dim=1)[:, 1]
        probs_celeb  = torch.softmax(celeb_model(images),  dim=1)[:, 1]

        stacked = torch.stack([probs_gan, probs_gan_ft, probs_celeb], dim=1)  # (B, 3)
        fake_max, winner_idx = stacked.max(dim=1)

        all_labels.extend(labels.tolist())
        all_fake_max.extend(fake_max.cpu().tolist())
        all_winner.extend(winner_idx.cpu().tolist())

all_labels = np.array(all_labels)
all_fake_max = np.array(all_fake_max)
all_winner = np.array(all_winner)

print(f"Collected scores for {len(all_labels)} images")

Using device: cuda
Test set size: 20000
✅ All three models loaded
Collected scores for 20000 images


In [ ]:
# ── ROC Curve ────────────────────────────────────────────────────────────
fpr, tpr, roc_thresholds = roc_curve(all_labels, all_fake_max)
roc_auc = auc(fpr, tpr)

# ── Precision-Recall Curve ───────────────────────────────────────────────
precision, recall, pr_thresholds = precision_recall_curve(all_labels, all_fake_max)

# ── Find optimal threshold using Youden's J statistic (TPR - FPR) ────────
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = roc_thresholds[optimal_idx]

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Optimal threshold (Youden's J): {optimal_threshold:.4f}")
print(f"At optimal threshold -> TPR: {tpr[optimal_idx]:.4f}, FPR: {fpr[optimal_idx]:.4f}")

# ── Compare against your default 0.5 threshold ───────────────────────────
preds_at_half = (all_fake_max >= 0.5).astype(int)
preds_at_optimal = (all_fake_max >= optimal_threshold).astype(int)

def quick_stats(preds, labels, name):
    tp = np.sum((preds == 1) & (labels == 1))
    tn = np.sum((preds == 0) & (labels == 0))
    fp = np.sum((preds == 1) & (labels == 0))
    fn = np.sum((preds == 0) & (labels == 1))
    acc = (tp + tn) / len(labels)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"\n{name}: TP={tp} TN={tn} FP={fp} FN={fn}")
    print(f"  Accuracy={acc:.4f}  Precision={prec:.4f}  Recall={rec:.4f}")

quick_stats(preds_at_half, all_labels, "Threshold = 0.5 (default)")
quick_stats(preds_at_optimal, all_labels, f"Threshold = {optimal_threshold:.4f} (optimal)")

# ── Plot both curves side by side ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].plot(fpr, tpr, color='#00d4ff', linewidth=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
axes[0].scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', s=80, zorder=5,
                label=f'Optimal (thr={optimal_threshold:.3f})')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve - PixelGuard Ensemble')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

axes[1].plot(recall, precision, color='#ff6b6b', linewidth=2)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve - PixelGuard Ensemble')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150, facecolor='white')
plt.show()
print("\n✅ Saved roc_pr_curves.png")